# Week 8 — XGBoost + Early Stopping + SHAP Explainability

**FraudFlow Project** | Week 8 Notebook

## Objectives
1. Train XGBClassifier with early stopping on an internal validation split
2. Handle class imbalance via `scale_pos_weight`
3. Evaluate on the untouched test set (threshold=0.5)
4. Generate SHAP explanations
5. Compare against previous models

> **Primary metric**: PR-AUC (Average Precision) — appropriate for severely imbalanced fraud detection.

## 1. Imports

In [ ]:
import json
import math
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, accuracy_score,
    confusion_matrix, precision_recall_curve, roc_curve,
)
from xgboost import XGBClassifier

import sys
sys.path.insert(0, '..')
from src.preprocessing.feature_engineering import engineer_features
from src.preprocessing.pipeline import build_preprocessor

RANDOM_STATE = 42
THRESHOLD = 0.5
SHAP_SAMPLE_SIZE = 5000
print('Imports complete.')

## 2. Load Data

In [ ]:
df_raw = pd.read_csv('../data/raw/fraudTest.csv')
print(f'Raw shape: {df_raw.shape}')
print(f'Fraud distribution:\n{df_raw["is_fraud"].value_counts()}')

## 3. Feature Engineering

In [ ]:
df = engineer_features(df_raw)

# Leakage guards
assert 'is_fraud' in df.columns
assert 'fraud_loss_amount' not in df.columns, 'Regression proxy must not be a feature!'

y = df['is_fraud']
X = df.drop(columns=['is_fraud'])

print(f'Feature matrix shape: {X.shape}')
print(f'Target distribution: {y.value_counts().to_dict()}')

## 4. Canonical Train/Test Split

- 80% train / 20% test
- Stratified on `is_fraud`
- `random_state=42`

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {len(y_train):,} rows | Fraud: {y_train.sum()} ({y_train.mean()*100:.4f}%)')
print(f'Test:  {len(y_test):,} rows  | Fraud: {y_test.sum()} ({y_test.mean()*100:.4f}%)')
print('Test set is now LOCKED — not used until final evaluation.')

## 4b. Preprocessing Pipeline

Preprocessor is fitted **only on training data**. Test data is only `transform()`-ed.

In [ ]:
preprocessor = build_preprocessor(X_train)
X_train_proc = preprocessor.fit_transform(X_train)  # fit + transform on train only
X_test_proc  = preprocessor.transform(X_test)       # transform only on test

# Extract meaningful feature names
num_cols  = preprocessor.transformers_[0][2]
cat_enc   = preprocessor.named_transformers_['cat']
cat_cols  = preprocessor.transformers_[1][2]
cat_names = cat_enc.get_feature_names_out(cat_cols)
feature_names = list(num_cols) + list(cat_names)

print(f'Total transformed features: {len(feature_names)}')
print(f'Sample feature names: {feature_names[:5]}')

# Dense conversion
if hasattr(X_train_proc, 'toarray'):
    X_train_proc = X_train_proc.toarray().astype('float32')
if hasattr(X_test_proc, 'toarray'):
    X_test_proc = X_test_proc.toarray().astype('float32')

print(f'X_train_proc shape: {X_train_proc.shape}')

## 5. Internal Validation Split

Created **only from training data** — the test set is never touched during early stopping.

In [ ]:
X_int_train, X_int_val, y_int_train, y_int_val = train_test_split(
    X_train_proc, y_train,
    test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)
print(f'Internal train: {len(y_int_train):,} rows | Internal val: {len(y_int_val):,} rows')

## 6. Class Imbalance — scale_pos_weight Calculation

Calculated **exclusively from the internal training set**, never from test data.

In [ ]:
neg_count = int((y_int_train == 0).sum())
pos_count = int((y_int_train == 1).sum())
scale_pos_weight = neg_count / pos_count

print(f'Negatives (legit, internal train):  {neg_count:,}')
print(f'Positives (fraud, internal train):  {pos_count:,}')
print(f'scale_pos_weight = {neg_count} / {pos_count} = {scale_pos_weight:.6f}')

## 7. XGBoost Configuration

In [ ]:
model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist',
    scale_pos_weight=scale_pos_weight,
    early_stopping_rounds=50,
)
print('Model configured.')
print(model)

## 8. Training with Early Stopping

Early stopping monitors `validation_1-aucpr` (internal validation set).
Training halts if no improvement for 50 consecutive rounds.

In [ ]:
model.fit(
    X_int_train, y_int_train,
    eval_set=[
        (X_int_train, y_int_train),
        (X_int_val,   y_int_val),
    ],
    verbose=50,
)

## 9. Validation Results (Early Stopping)

In [ ]:
best_iteration  = int(model.best_iteration)
best_val_pr_auc = float(model.best_score)
n_trees_used    = best_iteration + 1

print(f'Best iteration (0-indexed):  {best_iteration}')
print(f'Trees used:                  {n_trees_used}')
print(f'Best validation PR-AUC:      {best_val_pr_auc:.6f}')

## 10. Final Test Evaluation

The test set is evaluated **exactly once** after training. Threshold = 0.5 (no tuning).

In [ ]:
y_prob_test = model.predict_proba(X_test_proc)[:, 1]
y_pred_test = (y_prob_test >= THRESHOLD).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_test, target_names=['Legit', 'Fraud']))

pr_auc  = average_precision_score(y_test, y_prob_test)
roc_auc = roc_auc_score(y_test, y_prob_test)
print(f'PR-AUC (primary):  {pr_auc:.6f}')
print(f'ROC-AUC:           {roc_auc:.6f}')

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = cm.ravel()
print(f'TP={tp}  TN={tn}  FP={fp}  FN={fn}')

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('XGBoost Confusion Matrix (Week 8)')
plt.tight_layout()
plt.show()

## 12. Precision-Recall Curve

In [ ]:
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob_test)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall_vals, precision_vals, lw=2, label=f'XGBoost (AP={pr_auc:.4f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — XGBoost Week 8')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 13. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob_test)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(fpr, tpr, lw=2, label=f'XGBoost (AUC={roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — XGBoost Week 8')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 14. Previous-Model Comparison

Metrics loaded from stored Phase 3, 6, and 7 JSON files. **PR-AUC is the primary metric.**

In [ ]:
df_comp = pd.read_csv('../reports/phase8_model_comparison.csv')
print(df_comp.to_string(index=False))

metrics_to_plot = ['precision', 'recall', 'f1', 'pr_auc', 'roc_auc']
df_melt = df_comp.melt(id_vars='model', value_vars=metrics_to_plot,
                       var_name='metric', value_name='value').dropna()

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(metrics_to_plot))
width = 0.15
palette = plt.cm.tab10.colors
for i, (_, row) in enumerate(df_comp.iterrows()):
    vals = [row.get(m, 0) or 0 for m in metrics_to_plot]
    ax.bar(x + i * width, vals, width, label=row['model'], color=palette[i % len(palette)], alpha=0.85)

ax.set_xticks(x + width * (len(df_comp) - 1) / 2)
ax.set_xticklabels(['Precision', 'Recall', 'F1', 'PR-AUC*', 'ROC-AUC'])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison\n* PR-AUC is PRIMARY metric for imbalanced fraud detection')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 15. SHAP Analysis

Using `shap.TreeExplainer` — exact tree SHAP values. Sample of 5,000 test rows for speed.

In [ ]:
# Build DataFrame of test features
X_test_df = pd.DataFrame(X_test_proc, columns=feature_names)

# Reproducible sample
np.random.seed(RANDOM_STATE)
X_shap = X_test_df.sample(n=min(SHAP_SAMPLE_SIZE, len(X_test_df)), random_state=RANDOM_STATE)
X_shap = X_shap.reset_index(drop=True)
print(f'SHAP sample size: {len(X_shap)} rows')

# Compute SHAP values
explainer   = shap.TreeExplainer(model)
shap_values = explainer(X_shap)
print('SHAP computation complete.')

In [ ]:
# Global mean |SHAP| importance
vals = shap_values.values
if vals.ndim == 3:
    vals = vals[:, :, 1]
mean_abs_shap = np.abs(vals).mean(axis=0)
df_shap_imp = pd.DataFrame({'feature': feature_names, 'mean_abs_shap': mean_abs_shap})
df_shap_imp = df_shap_imp.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
print('Top 10 features by mean |SHAP|:')
print(df_shap_imp.head(10).to_string(index=False))

In [ ]:
# SHAP summary / beeswarm plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_shap, show=False, max_display=20)
plt.title('SHAP Summary Plot — XGBoost Week 8')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP bar plot (global feature importance)
plt.figure(figsize=(12, 8))
shap.plots.bar(shap_values, max_display=20, show=False)
plt.title('SHAP Bar Plot — Global Mean |SHAP| — XGBoost Week 8')
plt.tight_layout()
plt.show()

## 16. Conclusions

### Summary
- **XGBoost** with `scale_pos_weight` and early stopping on `aucpr` delivers strong PR-AUC performance on this severely imbalanced fraud dataset.
- Early stopping prevented overfitting by halting at the optimal number of boosting rounds.
- **SHAP** analysis reveals which features genuinely drive fraud predictions.

### Leakage Confirmation
- `is_fraud` — NOT in feature matrix ✅
- `fraud_loss_amount` (regression proxy) — NOT in feature matrix ✅
- Preprocessor fitted on training data only ✅
- Test set untouched during training and early stopping ✅
- `scale_pos_weight` computed from internal training set only ✅
- Threshold fixed at 0.5 (no tuning) ✅

### Next Steps (Week 9+)
- Week 9: Clustering / segmentation
- Week 10: Threshold optimization / cost-sensitive evaluation
- Week 11: Drift monitoring
- Week 12: Production deployment

> **Note**: This notebook is Week 8 only. No Week 9+ work is performed here.